In [ ]:
!pip install langchain langchain-community langchain-google-genai pageindex rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip install pageindex


In [ ]:
import os
from google.colab import userdata
from pageindex import PageIndexClient

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
pi_client = PageIndexClient(api_key=userdata.get('PAGEINDEX'))


In [ ]:
doc = pi_client.submit_document("full_corpus.pdf") #uploads the corpus PDF to PageIndex and generate a hierarchical document tree

In [ ]:
doc_id = doc["doc_id"]
print(doc_id)

pi-cmpwaom6e02jk01qui1i93l5g


In [ ]:
import time

while not pi_client.is_retrieval_ready(doc_id):
    print("Still indexing...")
    time.sleep(5)

print("Indexing complete!")

Still indexing...


In [ ]:
tree = pi_client.get_tree(
    doc_id,
    node_summary=True
)["result"]  #retrieve the generated hierarchical tree

In [ ]:
import pageindex.utils as utils

utils.print_tree(tree)

In [ ]:
from pprint import pprint

pprint(tree[0])

In [ ]:
documents = []

def traverse(node):

    documents.append({
        "title": node.get("title"),
        "node_id": node.get("node_id"),
        "page_index": node.get("page_index"),
        "text": node.get("text")
    })

    for child in node.get("nodes", []):
        traverse(child)

for root in tree:
    traverse(root)

In [ ]:
def chunk_text(text, size=200):

    words = text.split()

    chunks = []

    for i in range(0, len(words), size):
        chunks.append(
            " ".join(words[i:i+size])
        )

    return chunks

In [ ]:
all_chunks = []

for doc in documents:

    if doc["text"]:

        all_chunks.extend(
            chunk_text(doc["text"])
        )

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_chunks = [
    chunk.lower().split()
    for chunk in all_chunks
]

bm25 = BM25Okapi(tokenized_chunks)

In [ ]:
def retrieve(query, k=3):

    query_tokens = query.lower().split()

    scores = bm25.get_scores(query_tokens)

    ranked = sorted(
        zip(all_chunks, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked[:k]

In [ ]:
results = retrieve(
    "Why does Japan have hay fever?"
)

for i, (chunk, score) in enumerate(results, 1):

    print(f"\nResult {i}")
    print(f"Score: {score}")

    print(chunk[:500])


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(   #initialize Gemini model
    model="gemini-2.5-flash",
    temperature=0
)

In [ ]:
def rag_query(question):   #gives only the relevant chunks to the llm

    results = retrieve(question)

    context = "\n\n".join(
        chunk for chunk, score in results
    )

    prompt = f"""
Answer only using the context below.

Context:
{context}

Question:
{question}
"""

    response = llm.invoke(prompt)

    return response.content

In [ ]:
full_context = "\n\n".join(
    doc["text"]
    for doc in documents
    if doc["text"]
)

In [ ]:
def naive_query(question):  #gives full context to the llm

    prompt = f"""
Answer the question using the context below.

Context:
{full_context}

Question:
{question}
"""

    response = llm.invoke(prompt)

    return response.content


In [ ]:
import time

question = "Why does Japan have hay fever?"

start = time.time()

naive_answer = naive_query(question)

naive_time = time.time() - start

print("Naive Time:", naive_time)
print("\nAnswer:\n")
print(naive_answer)

start = time.time()

rag_answer = rag_query(question)

rag_time = time.time() - start

print("RAG Time:", rag_time)
print("\nAnswer:\n")
print(rag_answer)

Naive Time: 5.529360294342041

Answer:

Japan has hay fever primarily due to mass allergies driven by large amounts of pollen. This pollen comes from vast monoculture plantations of Japanese cedar (sugi) and Japanese cypress (hinoki) trees. These trees were extensively planted after World War Two as part of a rapid reforestation project to prevent soil erosion, following widespread deforestation during the war. The issue has become worse because these trees release more pollen after maturing, and most of them are now mature.
RAG Time: 4.716125726699829

Answer:

Japan has hay fever due to decisions made by leaders more than 70 years ago in the decades after World War Two. After widespread deforestation during the war, the government carried out large-scale afforestation, choosing to plant reams of only two different native, fast-growing evergreen species for rapid reforestation. Hay fever is driven by all the pollen from these trees.


In [ ]:
print(f"Naive Latency: {naive_time:.2f} seconds")
print(f"RAG Latency:   {rag_time:.2f} seconds")
#comparison between response time of both methods

Naive Latency: 5.53 seconds
RAG Latency:   4.72 seconds


In [ ]:
questions = [
    "Why does Japan have hay fever?",
    "Who was Estevanico?",
    "What is the AUKUS project?"
]

for q in questions:

    print("="*80)
    print("QUESTION:", q)

    print("\nNAIVE:")
    print(naive_query(q))

    print("\nRAG:")
    print(rag_query(q))

QUESTION: Why does Japan have hay fever?

NAIVE:
Japan has hay fever primarily due to mass allergies driven by large amounts of pollen. This pollen comes from vast monoculture plantations of Japanese cedar (sugi) and Japanese cypress (hinoki) trees. These trees were extensively planted after World War Two as part of a rapid reforestation project to prevent soil erosion, following widespread deforestation during the war. The issue has become worse because these trees release more pollen after maturing, and most of them are now mature.

RAG:
Japan has hay fever due to decisions made by leaders more than 70 years ago in the decades after World War Two. After widespread deforestation during the war, the government carried out large-scale afforestation, choosing to plant reams of only two different native, fast-growing evergreen species for rapid reforestation. Hay fever is driven by all the pollen from these trees.
QUESTION: Who was Estevanico?

NAIVE:
Estevanico was an enslaved Moroccan m